# METAR comparison to reanalysis products 

This notebook is set up to compare METAR to reanalysis products. The comparison is done with time series for a particular location first, then map-based comparisons for time periods. 

This notebook is set up to compare METAR with [AORC](https://registry.opendata.aws/noaa-nws-aorc/) data, a reanalysis product produced by NOAA for the continental U.S. and Alaska from 1979-2023. 

In [1]:
import matplotlib.pyplot as plt
import pandas as pd

from datetime import datetime,timedelta

import geopandas as gpd
from lonboard import viz, Map, ScatterplotLayer, HeatmapLayer
import duckdb

### Time series comparison 

Get the METAR data from [Dynamical](https://dynamical.org/). Choose a location and a time range. 

In [4]:
target = (39.742043, -104.991531)
target_METAR = 'DEN'

time_start = datetime(1940, 1, 1, 0)
time_end = datetime.now()

In [7]:
base = "https://data.source.coop/dynamical/asos-parquet"
urls = [f"{base}/year={y}/data.parquet" for y in range(time_start.year, time_end.year + 1)]
urls

['https://data.source.coop/dynamical/asos-parquet/year=1940/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1941/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1942/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1943/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1944/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1945/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1946/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1947/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1948/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1949/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1950/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1951/data.parquet',
 'https://data.source.coop/dynamical/asos-parquet/year=1952/data.parquet',
 'https://data.source.coo

In [16]:
df = duckdb.execute("""
    SELECT valid, longitude, latitude, station, name, country, tmpc, dwpc, relh
    FROM read_parquet($1, hive_partitioning=true)
    WHERE station = $2
    ORDER BY valid
""", [urls, target_METAR]).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [20]:
gdf = gpd.GeoDataFrame(df,geometry=gpd.points_from_xy(df.longitude,df.latitude))
gdf

,valid,longitude,latitude,station,name,country,tmpc,dwpc,relh,bbox,geometry
0,1974-11-13 03:00:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,0.56,-11.60,39.56,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
1,1974-11-13 06:00:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,1.11,-8.80,47.62,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
2,1974-11-13 09:00:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,-0.50,-11.00,44.95,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
3,1974-11-13 12:00:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,-1.00,-9.30,53.32,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
4,1974-11-13 15:00:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,-1.60,-8.20,60.72,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
...,...,...,...,...,...,...,...,...,...,...,...
329132,2026-06-16 11:53:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,9.44,-1.11,47.66,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
329133,2026-06-16 12:53:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,13.89,0.00,38.53,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
329134,2026-06-16 13:53:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,16.67,1.11,34.93,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)
329135,2026-06-16 14:53:00+00:00,-104.6575,39.8328,DEN,DENVER INTNL ARPT,US,20.00,1.67,29.50,"{'xmin': -104.6575, 'ymin': 39.8328, 'xmax': -...",POINT (-104.6575 39.8328)


In [12]:
df = duckdb.execute("""
    SELECT *
    FROM read_parquet($1, hive_partitioning=true)
    ORDER BY valid
""", [urls[1]]).fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [13]:
df.columns

Index(['station', 'valid', 'longitude', 'latitude', 'tmpf', 'tmpc', 'dwpf',
       'dwpc', 'relh', 'drct', 'sknt', 'gust', 'alti', 'mslp', 'vsby', 'p01i',
       'p01m', 'state', 'name', 'elevation', 'country', 'county', 'wfo',
       'tzname', 'geometry', 'bbox', 'year'],
      dtype='object')

In [ ]:
df = duckdb.execute("""
    SELECT *
    FROM read_parquet($1, hive_partitioning=true)
    WHERE 
---      country = 'FR' AND
    valid BETWEEN $2 AND $3
    ORDER BY country
""", [URLs, time_0, time_1]).fetchdf()